# Graph Neural Networks (message passing)

_Modern Deep Learning & AI_

**Each node updates itself by averaging its neighbours, then squishing. Do that a few times and the whole graph 'talks'.**

A graph is dots (nodes) joined by lines (edges). Think of friends and friendships, or atoms and bonds.

     A Graph Neural Network gives every node a small list of numbers (its features). Then it lets nodes share with their neighbours.

---

This notebook builds one round of message passing from scratch, piece by piece. Run each cell, read the note above it, and watch each node's features change as it averages in its neighbours. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We'll run a single graph-convolution (GCN) layer on a tiny 4-node graph. We build it in three parts: (1) the layer that mixes each node with its neighbours, (2) the graph itself as an adjacency matrix, (3) one forward pass that updates every node's features.

### Step 1 — Define one GCN layer

A GCN layer does two things to each node: **aggregate** then **transform**. Aggregation multiplies the row-normalized adjacency `A_hat` by the feature matrix `H`, which replaces each node's features with the *average* of its neighbours' features. The result is then passed through a shared linear layer `W` and a ReLU non-linearity — the same weights for every node.

In [ ]:
import torch
import torch.nn as nn

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)   # shared weight matrix W

    def forward(self, H, A_hat):
        # H: (N, in_dim) node features ; A_hat: (N, N) row-normalized adjacency
        agg = A_hat @ H                          # mean of each node's neighbours
        return torch.relu(self.lin(agg))        # sigma(W . agg)

### Step 2 — Build a tiny graph and its features

`A` is the adjacency matrix of a 4-node graph, including **self-loops** (the 1s on the diagonal) so each node keeps a bit of its own signal. Dividing every row by its sum turns those connections into averaging weights, giving the normalized `A_hat`. Each node also gets a random 3-dimensional feature vector to start from.

In [ ]:
# Tiny synthetic graph: 4 nodes, a couple of edges (+ self-loops on the diagonal).
A = torch.tensor([[1., 1., 0., 1.],
                  [1., 1., 1., 0.],
                  [0., 1., 1., 1.],
                  [1., 0., 1., 1.]])

A_hat = A / A.sum(dim=1, keepdim=True)   # normalize rows -> neighbour averaging
H = torch.randn(4, 3)                    # 4 nodes, 3 features each

### Step 3 — Run one round of message passing

Applying the layer once lets every node 'hear' from its immediate neighbours: the output `H2` holds each node's updated 2-dimensional feature vector. Stacking several such layers would let information travel further across the graph, one hop per layer.

In [ ]:
layer = GCNLayer(in_dim=3, out_dim=2)
H2 = layer(H, A_hat)                     # (4, 2) updated features
print(H2.shape)

## Visualize the data & results

_On the real Zachary karate-club graph, what happens to each node's features after one round of message passing?_

### Step 1 — Build the karate-club graph and node features

We load the real Zachary karate-club network (34 members, 0-indexed) from its edge list and fill a symmetric adjacency matrix `A`. For features we compute two genuinely structural quantities per node: its **degree** (how many neighbours it has) and its **triangle count** (how many of its neighbour pairs are themselves connected).

In [ ]:
import numpy as np

# Real Zachary karate-club edge list (Zachary 1977), 34 nodes, 0-indexed.
edges = [(0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),
(0,12),(0,13),(0,17),(0,19),(0,21),(0,31),(1,2),(1,3),(1,7),(1,13),(1,17),
(1,19),(1,21),(1,30),(2,3),(2,7),(2,8),(2,9),(2,13),(2,27),(2,28),(2,32),
(3,7),(3,12),(3,13),(4,6),(4,10),(5,6),(5,10),(5,16),(6,16),(8,30),(8,32),
(8,33),(9,33),(13,33),(14,32),(14,33),(15,32),(15,33),(18,32),(18,33),
(19,33),(20,32),(20,33),(22,32),(22,33),(23,25),(23,27),(23,29),(23,32),
(23,33),(24,25),(24,27),(24,31),(25,31),(26,29),(26,33),(27,33),(28,31),
(28,33),(29,32),(29,33),(30,32),(30,33),(31,32),(31,33),(32,33)]

n = 34
A = np.zeros((n, n))
for i, j in edges:
    A[i, j] = 1.0
    A[j, i] = 1.0

# Two REAL structural node features: degree and triangle count.
deg = A.sum(1)
tri = np.array([(A @ A * A)[i].sum() / 2 for i in range(n)])
H = np.column_stack([deg, tri])

### Step 2 — Apply one GCN layer (averaging step only)

Here we run just the aggregation half of a GCN layer: add self-loops, row-normalize, and multiply by the features. This replaces each node's `[degree, triangles]` with the average of its own values and those of its neighbours — the heart of message passing, with no learned weights so we can read the effect directly.

In [ ]:
# One GCN layer = mean over self + neighbours (symmetric self-loop, row-normalized).
A_hat = A + np.eye(n)
A_hat = A_hat / A_hat.sum(1, keepdims=True)
H2 = A_hat @ H

### Step 3 — Compare features before and after

We pick six interesting nodes (including the two faction leaders, 'Mr Hi' and the 'Officer') and show their two features side by side. The BEFORE panel shows the raw degree and triangle counts; the AFTER panel shows the smoothed values once each node has mixed in its neighbours. Notice how the numbers get pulled toward a local average.

In [ ]:
sel = [0, 33, 1, 2, 32, 3]
names = ['Mr Hi (0)', 'Officer (33)', 'Node 1', 'Node 2', 'Node 32', 'Node 3']

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(8, 4))

panels = [(ax[0], H[sel], 'BEFORE'), (ax[1], H2[sel], 'AFTER one GCN layer')]
for a, M, t in panels:
    a.imshow(M, cmap='viridis', aspect='auto')
    a.set_xticks([0, 1])
    a.set_xticklabels(['degree', 'triangles'])
    a.set_yticks(range(6))
    a.set_yticklabels(names)
    a.set_title(t)
    for i in range(6):
        for j in range(2):
            a.text(j, i, format(M[i, j], '.2f'), ha='center', va='center', color='w')

plt.tight_layout()
plt.show()